# 08 — LoRA e QLoRA no núcleo do Lab-IA

O caderno 02 usou PEFT/bitsandbytes para LoRA/QLoRA. O Lab-IA tem implementações
próprias (`labia.models.lora`, `labia.models.quant`) para você ver o mecanismo sem
camada de biblioteca: base congelada + ramo de baixo posto *B·A*, quantização int8/NF4
e fusão dos pesos no fim.

Pré-requisito: caderno 01 executado (mesmo tokenizer e corpus do caderno 07).


In [ ]:
import copy
from pathlib import Path
import torch
from tokenizers import Tokenizer

from labia.models.gpt import ConfigGPT, GPT
from labia.models.lora import (
    aplicar_lora, carregar_adaptador, mesclar_lora, salvar_adaptador,
)
from labia.trainer.dados import montar_dataset

ARQ_TOKENIZER = Path('artifacts/tokenizer_puro/tokenizer.json')
if not ARQ_TOKENIZER.is_file():
    raise FileNotFoundError('Tokenizer ausente. Execute primeiro o caderno 01.')
tok = Tokenizer.from_file(str(ARQ_TOKENIZER))
corpus = Path('data/corpus.txt').read_text(encoding='utf-8')

JANELA = 128
torch.manual_seed(42)
config = ConfigGPT(vocab=tok.get_vocab_size(), dim=128, camadas=4, cabecas=4,
                   janela_ctx=JANELA, abandono=0.0, norm='rmsnorm', pos='rope', mlp='swiglu')
modelo = GPT(config)
modelo.init_pesos(semente=42)
disp = 'cuda' if torch.cuda.is_available() else 'cpu'
modelo = modelo.to(disp)

def treinar(modelo_alvo, x, y, passos=60, lote=16, lr=5e-4):
    treinaveis = [p for p in modelo_alvo.parameters() if p.requires_grad]
    otim = torch.optim.AdamW(treinaveis, lr=lr)
    modelo_alvo.train()
    for passo in range(passos):
        ini = (passo * lote) % max(1, len(x) - lote)
        _, perda = modelo_alvo(x[ini:ini + lote].to(disp), y[ini:ini + lote].to(disp))
        otim.zero_grad(set_to_none=True)
        perda.backward()
        otim.step()
    modelo_alvo.eval()
    return float(perda)

@torch.no_grad()
def ce_do(modelo_alvo, x, y, lote=32):
    total, n = 0.0, 0
    for i in range(0, len(x), lote):
        _, p = modelo_alvo(x[i:i + lote].to(disp), y[i:i + lote].to(disp))
        total += float(p); n += 1
    return total / max(1, n)

x_trem, y_trem = montar_dataset(tok, corpus, JANELA, stride=JANELA // 2)
treinar(modelo, x_trem, y_trem, passos=120)
estado_base = copy.deepcopy(modelo.state_dict())
print('base densa treinada; valência do próximo experimento:', round(ce_do(modelo, x_trem, y_trem), 3))


## 1. Domínio novo, três caminhos de ajuste

O "domínio novo" abaixo são regras de lógica proposicional fora do corpus original.
Três estratégias com o mesmo orçamento de passos:

| Estratégia | O que treina |
|---|---|
| **A. Full fine-tuning** | todos os pesos |
| **B. LoRA** | só os ramos A/B injetados (base congelada) |
| **C. QLoRA** | idem, mas a base vive quantizada em **NF4** |

Medimos as duas coisas que interessam: aprender o domínio novo **e** não esquecer o corpus.

In [ ]:
regras = '\n'.join([
    'Premissa 1: se chover entao a rua molha.',
    'Premissa 2: choveu.',
    'Conclusao: a rua esta molha. (modus ponens)',
    'Premissa 1: se estudar entao aprender.',
    'Premissa 2: nao aprendeu.',
    'Conclusao: nao estudou. (modus tollens)',
    'Se A implica B e B implica C, entao A implica C.',
    'Nao e verdade que A e nao A.',
]) * 6
x_novo, y_novo = montar_dataset(tok, regras, JANELA, stride=JANELA // 2)

def fabricar():
    m = GPT(config).to(disp)
    m.load_state_dict(estado_base)
    return m

resultados = {}

# A — full fine-tuning
m = fabricar()
treinar(m, x_novo, y_novo, passos=60)
resultados['A_full'] = (ce_do(m, x_novo, y_novo), ce_do(m, x_trem, y_trem),
                        sum(p.numel() for p in m.parameters() if p.requires_grad))

# B — LoRA
m = fabricar()
stats = aplicar_lora(m, r=8, alpha=16)
treinar(m, x_novo, y_novo, passos=60, lr=1e-3)
resultados['B_lora'] = (ce_do(m, x_novo, y_novo), ce_do(m, x_trem, y_trem), stats['treinaveis'])

# C — QLoRA (base NF4 congelada)
m = fabricar()
stats = aplicar_lora(m, r=8, alpha=16, quant='nf4')
treinar(m, x_novo, y_novo, passos=60, lr=1e-3)
resultados['C_qlora'] = (ce_do(m, x_novo, y_novo), ce_do(m, x_trem, y_trem), stats['treinaveis'])

total_param = sum(t.numel() for t in estado_base.values())
print(f"{'estratégia':<8} {'CE domínio novo':>16} {'CE corpus (esquecimento)':>26} {'parâmetros treináveis':>24}")
for nome, (novo, antigo, trein) in resultados.items():
    print(f'{nome:<8} {novo:>16.3f} {antigo:>26.3f} {trein:>18,} ({trein / total_param:.1%})')


## 2. Salvar e recarregar o adaptador

O checkpoint guarda só os ramos A/B treináveis (e os buffers quantizados, se houver).
Isto vem **antes** da fusão: depois de fundir, não existe mais adaptador para salvar.

In [ ]:
pasta_adaptador = Path('artifacts/lora_labia')
m = fabricar()
aplicar_lora(m, r=8, alpha=16)
treinar(m, x_novo, y_novo, passos=60, lr=1e-3)
salvar_adaptador(m, pasta_adaptador, meta_extra={'dominio': 'regras-logicas'})

m_recuperado = fabricar()
aplicar_lora(m_recuperado, r=8, alpha=16)
meta = carregar_adaptador(m_recuperado, pasta_adaptador)
print('meta do adaptador:', {k: meta[k] for k in ('r', 'alpha', 'quant', 'treinaveis')})
print('CE identico após recarregar:', abs(ce_do(m_recuperado, x_novo, y_novo) - ce_do(m, x_novo, y_novo)) < 1e-6)


## 3. Fundir o adaptador (o `merge_and_unload` do caderno 02)

A fusão aplica `W ← W + escala·(B·A)` e devolve lineares puros: treino termina,
inferência sem custo de ramo extra.

In [ ]:
amostra = x_trem[:4].to(disp)
antes = m(amostra)[0].clone()
trocados = mesclar_lora(m)
depois = m(amostra)[0]
print(f'{trocados} lineares fundidos | maior diferença antes/depois: '
      f'{(antes - depois).abs().max().item():.2e} (esperado: ~0, só arredondamento)')


## Correspondência com o caderno 02

| Caderno 02 (bibliotecas) | Aqui (núcleo Lab-IA) |
|---|---|
| `LoraConfig` + `get_peft_model` | `aplicar_lora(modelo, r, alpha)` |
| `BitsAndBytesConfig(load_in_4bit=True)` | `aplicar_lora(..., quant='nf4')` |
| `merge_and_unload()` | `mesclar_lora(modelo)` |
| `save_pretrained` / `PeftModel.from_pretrained` | `salvar_adaptador` / `carregar_adaptador` |

A quantização nativa é mais simples que a do bitsandbytes (sem double-quant nem
paginação) — suficiente para estudo; o caderno 02 continua sendo o caminho de produção.

## Exercícios

1. Rode B e C com `r=1` e `r=32`: quanto de posto basta para este domínio?
2. Troque `quant='nf4'` por `'int8'` e compare tamanho real (`labia.models.quant.tamanho_state_dict`).
3. Amplie `alvos` além de `ALVOS_PADRAO` e observe o efeito nos parâmetros treináveis.